In [ ]:
import os
import sys

import numpy as np
from reachy_sdk import ReachySDK
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode

sys.path.insert(0, os.path.abspath("../src"))  # this repo's src/, for reachy_ai
from reachy_ai.sdk_compat import fix_command_poll  # noqa: E402

# reachy_sdk 0.7.0 leaks memory on every command push unless this runs before ReachySDK() (issue #23)
print("reachy_sdk command-poll fix applied:", fix_command_poll())
reachy = ReachySDK(host=os.environ["REACHY_IP"])

In [ ]:
reachy

In [ ]:
pose = reachy.r_arm.forward_kinematics()
print(pose)

In [ ]:
def show_position(name):
    pose = reachy.r_arm.forward_kinematics()

    x = pose[0, 3]
    y = pose[1, 3]
    z = pose[2, 3]

    print(
        f'{name}: '
        f'({x:.4f}, {y:.4f}, {z:.4f})'
    )

    return pose

In [ ]:
SAFE_TRAVEL_OFFSET = 0.08   # 8 cm above taught pose
MIN_BOARD_OFFSET   = 0.03   # NEVER descend below this offset

def get_board_pose(square, offset):
    if offset < MIN_BOARD_OFFSET:
        raise ValueError(
            f"Unsafe Z offset: {offset:.3f} m. "
            f"Minimum allowed is {MIN_BOARD_OFFSET:.3f} m."
        )

    pose = BOARD_POSES[square].copy()
    pose[2, 3] += offset

    return pose

In [ ]:
#show_position("A1")

In [ ]:

BOARD_POSES = {

    "A1": np.array([
        [-0.01808295, -0.81483479, -0.57941114,  0.34214051],
        [ 0.05606442,  0.57776777, -0.81427341,  0.11797679],
        [ 0.99826339, -0.04720881,  0.03523549, -0.37703476],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "A2": np.array([
        [-0.00578759, -0.31373665, -0.94949240,  0.34933746],
        [ 0.01651412,  0.94934883, -0.31378987,  0.01736093],
        [ 0.99984688, -0.01749612, -0.00031335, -0.37566538],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "A3": np.array([
        [ 0.01196106, -0.11503405, -0.99328953,  0.35621112],
        [ 0.04124568,  0.99257192, -0.11445427, -0.12895906],
        [ 0.99907744, -0.03959990,  0.01661687, -0.37692032],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "B1": np.array([
        [-0.00487647, -0.70553550, -0.70865780,  0.23123656],
        [-0.05203516,  0.70788517, -0.70440821,  0.14558051],
        [ 0.99863335,  0.03344010, -0.04016464, -0.37192733],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "B2": np.array([
        [-0.00017153, -0.69616953, -0.71787740,  0.24251293],
        [ 0.11818002,  0.71283254, -0.69130547,  0.00612421],
        [ 0.99299217, -0.08495735,  0.08215105, -0.38243731],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "B3": np.array([
        [-0.09060166, -0.05718291, -0.99424416,  0.23253929],
        [-0.09890699,  0.99393099, -0.04815188, -0.14153537],
        [ 0.99096354,  0.09397506, -0.09570759, -0.36954387],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "C1": np.array([
        [-0.01729617, -0.97149490, -0.23642865,  0.10316200],
        [ 0.03139282,  0.23581978, -0.97128962,  0.14830975],
        [ 0.99935746, -0.02422175,  0.02641918, -0.37783482],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "C2": np.array([
        [-0.06127873, -0.98928949, -0.13248105,  0.10070317],
        [ 0.08789909,  0.12686606, -0.98801759, -0.00684353],
        [ 0.99424276, -0.07218942,  0.07918345, -0.38245479],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ]),

    "C3": np.array([
        [ 0.00572237, -0.25876516, -0.96592331,  0.07757317],
        [ 0.03171251,  0.96550023, -0.25846395, -0.15544926],
        [ 0.99948065, -0.02915282,  0.01373105, -0.37839183],
        [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
    ])
}

In [ ]:
import numpy as np

target_pose = BOARD_POSES["B2"].copy()

# Raise gripper 8 cm above the recorded position
target_pose[2, 3] += 0.08

print("B2 recorded pose:")
print(BOARD_POSES["B2"])

print("\nB2 approach pose:")
print(target_pose)

In [ ]:
# B2 approach pose should already exist:
# target_pose = BOARD_POSES["B2"].copy()
# target_pose[2, 3] += 0.08

try:
    joint_positions = reachy.r_arm.inverse_kinematics(
        target_pose
    )

    print("IK solution found:")
    print(joint_positions)

except ValueError as e:
    print("No IK solution found:")
    print(e)

In [ ]:
joint_positions = [
    20.449441607703104,
    14.065440168202297,
    18.083163533168882,
    -96.09435552565053,
    25.80714537579904,
    -7.480792185500247,
    27.053098670552206
]

calculated_pose = reachy.r_arm.forward_kinematics(
    joint_positions
)

print("Requested B2 approach pose:")
print(target_pose)

print("\nFK result:")
print(calculated_pose)

print("\nRequested XYZ:")
print(
    target_pose[0,3],
    target_pose[1,3],
    target_pose[2,3]
)

print("\nCalculated XYZ:")
print(
    calculated_pose[0,3],
    calculated_pose[1,3],
    calculated_pose[2,3]
)

In [ ]:
reachy.turn_on('reachy')

In [ ]:
b2_approach_joints = {
    reachy.r_arm.r_shoulder_pitch: joint_positions[0],
    reachy.r_arm.r_shoulder_roll:  joint_positions[1],
    reachy.r_arm.r_arm_yaw:        joint_positions[2],
    reachy.r_arm.r_elbow_pitch:    joint_positions[3],
    reachy.r_arm.r_forearm_yaw:    joint_positions[4],
    reachy.r_arm.r_wrist_pitch:    joint_positions[5],
    reachy.r_arm.r_wrist_roll:     joint_positions[6],
}

print("Moving to B2 approach position...")

goto(
    goal_positions=b2_approach_joints,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Movement complete.")

In [ ]:
actual_pose = reachy.r_arm.forward_kinematics()

print("Actual final pose:")
print(actual_pose)

print("\nActual XYZ:")
print(
    actual_pose[0,3],
    actual_pose[1,3],
    actual_pose[2,3]
)

In [ ]:
print("COMMAND vs ACTUAL\n")

joint_names = [
    "shoulder_pitch",
    "shoulder_roll",
    "arm_yaw",
    "elbow_pitch",
    "forearm_yaw",
    "wrist_pitch",
    "wrist_roll"
]

motors = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll
]

for name, motor, target in zip(
    joint_names,
    motors,
    joint_positions
):
    actual = motor.present_position

    print(
        f"{name:16s} "
        f"target={target:8.2f}°  "
        f"actual={actual:8.2f}°  "
        f"error={actual-target:+7.2f}°"
    )

In [ ]:
# Start with the original recorded B2 pose
b2_test_pose = BOARD_POSES["B2"].copy()

# Stop 3 cm above the taught B2 position
b2_test_pose[2, 3] += 0.03

print("B2 test pose:")
print(b2_test_pose)

# Calculate IK
b2_test_joints = reachy.r_arm.inverse_kinematics(
    b2_test_pose
)

print("\nIK solution:")
print(b2_test_joints)

In [ ]:
b2_test_goal = {
    reachy.r_arm.r_shoulder_pitch: b2_test_joints[0],
    reachy.r_arm.r_shoulder_roll:  b2_test_joints[1],
    reachy.r_arm.r_arm_yaw:        b2_test_joints[2],
    reachy.r_arm.r_elbow_pitch:    b2_test_joints[3],
    reachy.r_arm.r_forearm_yaw:    b2_test_joints[4],
    reachy.r_arm.r_wrist_pitch:    b2_test_joints[5],
    reachy.r_arm.r_wrist_roll:     b2_test_joints[6],
}

goto(
    goal_positions=b2_test_goal,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached B2 test position.")

In [ ]:
# Return to the safer B2 approach position
b2_safe_pose = BOARD_POSES["B2"].copy()
b2_safe_pose[2, 3] += 0.08

b2_safe_joints = reachy.r_arm.inverse_kinematics(
    b2_safe_pose
)

b2_safe_goal = {
    reachy.r_arm.r_shoulder_pitch: b2_safe_joints[0],
    reachy.r_arm.r_shoulder_roll:  b2_safe_joints[1],
    reachy.r_arm.r_arm_yaw:        b2_safe_joints[2],
    reachy.r_arm.r_elbow_pitch:    b2_safe_joints[3],
    reachy.r_arm.r_forearm_yaw:    b2_safe_joints[4],
    reachy.r_arm.r_wrist_pitch:    b2_safe_joints[5],
    reachy.r_arm.r_wrist_roll:     b2_safe_joints[6],
}

goto(
    goal_positions=b2_safe_goal,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
right_angled_position_right = {
    reachy.r_arm.r_shoulder_pitch: -55,
    reachy.r_arm.r_shoulder_roll:  0,
    reachy.r_arm.r_arm_yaw:        45,
    reachy.r_arm.r_elbow_pitch:    -90,
    reachy.r_arm.r_forearm_yaw:    0,
    reachy.r_arm.r_wrist_pitch:    0,
    reachy.r_arm.r_wrist_roll:     -45,
}

goto(
    goal_positions=right_angled_position_right,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
# Create safe B2 approach pose
b2_safe_pose = get_board_pose("B2", 0.08)

# Calculate joint positions
b2_safe_joints = reachy.r_arm.inverse_kinematics(
    b2_safe_pose
)

b2_safe_goal = {
    reachy.r_arm.r_shoulder_pitch: b2_safe_joints[0],
    reachy.r_arm.r_shoulder_roll:  b2_safe_joints[1],
    reachy.r_arm.r_arm_yaw:        b2_safe_joints[2],
    reachy.r_arm.r_elbow_pitch:    b2_safe_joints[3],
    reachy.r_arm.r_forearm_yaw:    b2_safe_joints[4],
    reachy.r_arm.r_wrist_pitch:    b2_safe_joints[5],
    reachy.r_arm.r_wrist_roll:     b2_safe_joints[6],
}

print("Moving above B2...")

goto(
    goal_positions=b2_safe_goal,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached B2 approach position.")

In [ ]:
b2_safe_taught = reachy.r_arm.forward_kinematics()

print("B2 SAFE taught pose:")
print(b2_safe_taught)

print("\nXYZ:")
print(
    b2_safe_taught[0,3],
    b2_safe_taught[1,3],
    b2_safe_taught[2,3]
)

In [ ]:
B2_SAFE = np.array([
    [ 0.05235371, -0.19250409, -0.97989860,  0.27044869],
    [ 0.01557489,  0.98128233, -0.19194379,  0.01176236],
    [ 0.99850714, -0.00521285,  0.05437200, -0.29502528],
    [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
])

In [ ]:
B2_PICK = reachy.r_arm.forward_kinematics()

print("B2 PICK taught pose:")
print(B2_PICK)

print("\nXYZ:")
print(
    B2_PICK[0,3],
    B2_PICK[1,3],
    B2_PICK[2,3]
)

In [ ]:
print(
    "Open gripper position:",
    reachy.r_arm.r_gripper.present_position
)

In [ ]:
GRIPPER_OPEN   = -29.79
GRIPPER_GRASP  = -18.94
GRIPPER_CLOSED =  12.08

In [ ]:
B2_PICK = np.array([
    [ 0.22626608, -0.19838134, -0.95365009,  0.22172797],
    [ 0.02680319,  0.97993831, -0.19749049,  0.01172544],
    [ 0.97369669,  0.01912454,  0.22704406, -0.34096297],
    [ 0.00000000,  0.00000000,  0.00000000,  1.00000000]
])

In [ ]:
B2_PICK = reachy.r_arm.forward_kinematics()

print("B2 PICK taught pose:")
print(B2_PICK)

print("\nXYZ:")
print(
    B2_PICK[0,3],
    B2_PICK[1,3],
    B2_PICK[2,3]
)

In [ ]:
GRIPPER_GRASP = reachy.r_arm.r_gripper.present_position

print("Cube grasp position:", GRIPPER_GRASP)

In [ ]:
reachy.turn_on('r_arm')

In [ ]:
def move_to_pose(pose, duration=4.0):

    joints = reachy.r_arm.inverse_kinematics(pose)

    goal = {
        reachy.r_arm.r_shoulder_pitch: joints[0],
        reachy.r_arm.r_shoulder_roll:  joints[1],
        reachy.r_arm.r_arm_yaw:        joints[2],
        reachy.r_arm.r_elbow_pitch:    joints[3],
        reachy.r_arm.r_forearm_yaw:    joints[4],
        reachy.r_arm.r_wrist_pitch:    joints[5],
        reachy.r_arm.r_wrist_roll:     joints[6],
    }

    goto(
        goal_positions=goal,
        duration=duration,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

In [ ]:
print("Moving to B2 SAFE...")
move_to_pose(B2_SAFE, 5.0)

print("STOP HERE")

In [ ]:
# ============================================
# REACHY RIGHT ARM RESET POSITION
# ============================================

RESET_POSITION_RIGHT = {
    reachy.r_arm.r_shoulder_pitch: -55,
    reachy.r_arm.r_shoulder_roll:    0,
    reachy.r_arm.r_arm_yaw:         45,
    reachy.r_arm.r_elbow_pitch:    -90,
    reachy.r_arm.r_forearm_yaw:      0,
    reachy.r_arm.r_wrist_pitch:      0,
    reachy.r_arm.r_wrist_roll:     -45,
}


def reset_right_arm():

    print("Moving right arm to RESET position...")

    # Make arm stiff/active
    reachy.turn_on("r_arm")

    goto(
        goal_positions=RESET_POSITION_RIGHT,
        duration=5.0,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

    print("Right arm is at RESET position.")

In [ ]:
reset_right_arm()

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
    reachy.r_arm.r_gripper,
]

for joint in joints:
    print(
        joint.name,
        "| compliant =", joint.compliant,
        "| present =", joint.present_position,
        "| goal =", joint.goal_position
    )

In [ ]:
reachy.turn_off("r_arm")

In [ ]:
print("Arm compliant states:")

for joint in [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
    reachy.r_arm.r_gripper,
]:
    print(
        joint.name,
        "| compliant =", joint.compliant,
        "| present =", joint.present_position
    )

In [ ]:
print(
    "Shoulder pitch:",
    reachy.r_arm.r_shoulder_pitch.present_position
)

print(
    "Shoulder roll:",
    reachy.r_arm.r_shoulder_roll.present_position
)

In [ ]:
for joint in [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]:
    print(
        f"{joint.name:18s}",
        f"{joint.present_position:7.2f}"
    )

In [ ]:
arm_joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

# 1. Read current positions
current_positions = [
    joint.present_position
    for joint in arm_joints
]

print("Current positions:")
print(current_positions)

# 2. Make current physical positions the goals
for joint, position in zip(
    arm_joints,
    current_positions
):
    joint.goal_position = position

# 3. Turn arm stiff at those positions
reachy.turn_on("r_arm")

print("Arm is now stiff at current position.")

In [ ]:
RESET_POSITION_RIGHT = {
    reachy.r_arm.r_shoulder_pitch: -25.09,
    reachy.r_arm.r_shoulder_roll:  -21.31,
    reachy.r_arm.r_arm_yaw:        -13.36,
    reachy.r_arm.r_elbow_pitch:   -116.81,
    reachy.r_arm.r_forearm_yaw:     -3.34,
    reachy.r_arm.r_wrist_pitch:      28.92,
    reachy.r_arm.r_wrist_roll:       32.26,
}

In [ ]:
def reset_right_arm():

    print("Moving right arm to RESET...")

    goto(
        goal_positions=RESET_POSITION_RIGHT,
        duration=6.0,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

    print("Right arm reset complete.")

In [ ]:
reset_right_arm()

In [ ]:
RESET_POSITION_RIGHT = {
    reachy.r_arm.r_shoulder_pitch:
        reachy.r_arm.r_shoulder_pitch.present_position,

    reachy.r_arm.r_shoulder_roll:
        reachy.r_arm.r_shoulder_roll.present_position,

    reachy.r_arm.r_arm_yaw:
        reachy.r_arm.r_arm_yaw.present_position,

    reachy.r_arm.r_elbow_pitch:
        reachy.r_arm.r_elbow_pitch.present_position,

    reachy.r_arm.r_forearm_yaw:
        reachy.r_arm.r_forearm_yaw.present_position,

    reachy.r_arm.r_wrist_pitch:
        reachy.r_arm.r_wrist_pitch.present_position,

    reachy.r_arm.r_wrist_roll:
        reachy.r_arm.r_wrist_roll.present_position,
}

for joint, position in RESET_POSITION_RIGHT.items():
    print(joint.name, position)

In [ ]:
HOME_POSITION = {
    reachy.r_arm.r_shoulder_pitch:  11.56,
    reachy.r_arm.r_shoulder_roll:  -20.17,
    reachy.r_arm.r_arm_yaw:        -12.74,
    reachy.r_arm.r_elbow_pitch:   -115.84,
    reachy.r_arm.r_forearm_yaw:     -3.52,
    reachy.r_arm.r_wrist_pitch:      29.62,
    reachy.r_arm.r_wrist_roll:       32.61,
}

In [ ]:
def go_home():

    print("Moving to HOME...")

    goto(
        goal_positions=HOME_POSITION,
        duration=6.0,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

    print("HOME reached.")

In [ ]:
reachy.turn_off("r_arm")

In [ ]:
arm_joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

B2_APPROACH_VALUES = [
    joint.present_position
    for joint in arm_joints
]

print("B2 APPROACH joint positions:")

for joint, value in zip(
    arm_joints,
    B2_APPROACH_VALUES
):
    print(f"{joint.name:18s} {value:7.2f}")

In [ ]:
B2_APPROACH = {
    reachy.r_arm.r_shoulder_pitch:   6.90,
    reachy.r_arm.r_shoulder_roll:   -7.69,
    reachy.r_arm.r_arm_yaw:         32.61,
    reachy.r_arm.r_elbow_pitch:    -85.87,
    reachy.r_arm.r_forearm_yaw:     -8.96,
    reachy.r_arm.r_wrist_pitch:    -20.21,
    reachy.r_arm.r_wrist_roll:      54.14,
}

In [ ]:
# ==========================================
# RECOVER -> HOME -> B2 APPROACH
# ==========================================

import time

arm_joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

# --------------------------------------------------
# Known physically verified positions
# --------------------------------------------------

HOME_POSITION = {
    reachy.r_arm.r_shoulder_pitch:  11.56,
    reachy.r_arm.r_shoulder_roll:  -20.17,
    reachy.r_arm.r_arm_yaw:        -12.74,
    reachy.r_arm.r_elbow_pitch:   -115.84,
    reachy.r_arm.r_forearm_yaw:     -3.52,
    reachy.r_arm.r_wrist_pitch:      29.62,
    reachy.r_arm.r_wrist_roll:       32.61,
}

B2_APPROACH = {
    reachy.r_arm.r_shoulder_pitch:   6.90,
    reachy.r_arm.r_shoulder_roll:   -7.69,
    reachy.r_arm.r_arm_yaw:         32.61,
    reachy.r_arm.r_elbow_pitch:    -85.87,
    reachy.r_arm.r_forearm_yaw:     -8.96,
    reachy.r_arm.r_wrist_pitch:    -20.21,
    reachy.r_arm.r_wrist_roll:      54.14,
}


# --------------------------------------------------
# STEP 1: Capture wherever the arm is RIGHT NOW
# --------------------------------------------------

print("Synchronizing current position...")

for joint in arm_joints:
    joint.goal_position = joint.present_position


# --------------------------------------------------
# STEP 2: Make arm stiff AT CURRENT POSITION
# --------------------------------------------------

print("Making arm stiff...")

reachy.turn_on("r_arm")

time.sleep(2)

print("Arm should now be holding its current position.")

In [ ]:
print("Moving slowly to HOME...")

goto(
    goal_positions=HOME_POSITION,
    duration=8.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("HOME reached.")

In [ ]:
print("Moving slowly to B2 APPROACH...")

goto(
    goal_positions=B2_APPROACH,
    duration=8.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("B2 APPROACH reached.")

In [ ]:
SAFE_HEIGHT = 0.15  # 15 cm above taught board position

def get_safe_pose(square):
    pose = BOARD_POSES[square].copy()
    pose[2, 3] += SAFE_HEIGHT
    return pose

In [ ]:
for square in ["A1","A2","A3",
               "B1","B2","B3",
               "C1","C2","C3"]:

    pose = get_safe_pose(square)

    print(
        square,
        "XYZ:",
        round(pose[0,3], 3),
        round(pose[1,3], 3),
        round(pose[2,3], 3)
    )

In [ ]:
b2_safe = get_safe_pose("B2")

b2_safe_joints = reachy.r_arm.inverse_kinematics(
    b2_safe
)

print("B2 safe XYZ:",
      b2_safe[0,3],
      b2_safe[1,3],
      b2_safe[2,3])

print("B2 safe joints:")
print(b2_safe_joints)

In [ ]:
goto(
    goal_positions=HOME_POSITION,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("HOME reached.")

In [ ]:
B2_SAFE_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  23.4794,
    reachy.r_arm.r_shoulder_roll:   20.2843,
    reachy.r_arm.r_arm_yaw:         19.9217,
    reachy.r_arm.r_elbow_pitch:   -111.0259,
    reachy.r_arm.r_forearm_yaw:     32.5216,
    reachy.r_arm.r_wrist_pitch:       4.5934,
    reachy.r_arm.r_wrist_roll:       34.5556,
}

In [ ]:
print("Moving to B2 SAFE (+15 cm)...")

goto(
    goal_positions=B2_SAFE_JOINTS,
    duration=8.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("B2 SAFE reached.")

In [ ]:
SAFE_HEIGHT = 0.05

def get_safe_pose(square):
    pose = BOARD_POSES[square].copy()
    pose[2, 3] += SAFE_HEIGHT
    return pose


def move_to_safe_square(square, duration=8.0):

    print(f"Moving safely above {square}...")

    pose = get_safe_pose(square)

    joints = reachy.r_arm.inverse_kinematics(pose)

    goal = {
        reachy.r_arm.r_shoulder_pitch: joints[0],
        reachy.r_arm.r_shoulder_roll:  joints[1],
        reachy.r_arm.r_arm_yaw:        joints[2],
        reachy.r_arm.r_elbow_pitch:    joints[3],
        reachy.r_arm.r_forearm_yaw:    joints[4],
        reachy.r_arm.r_wrist_pitch:    joints[5],
        reachy.r_arm.r_wrist_roll:     joints[6],
    }

    goto(
        goal_positions=goal,
        duration=duration,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

    print(f"Reached safe position above {square}.")

In [ ]:
go_home()

move_to_safe_square("B2")

In [ ]:
B2_ROTATED = B2_SAFE_JOINTS.copy()

B2_ROTATED[reachy.r_arm.r_wrist_roll] = (
    B2_SAFE_JOINTS[reachy.r_arm.r_wrist_roll] + 45
)

goto(
    goal_positions=B2_ROTATED,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
B2_SAFE_ALIGNED = {
    reachy.r_arm.r_shoulder_pitch:
        reachy.r_arm.r_shoulder_pitch.present_position,

    reachy.r_arm.r_shoulder_roll:
        reachy.r_arm.r_shoulder_roll.present_position,

    reachy.r_arm.r_arm_yaw:
        reachy.r_arm.r_arm_yaw.present_position,

    reachy.r_arm.r_elbow_pitch:
        reachy.r_arm.r_elbow_pitch.present_position,

    reachy.r_arm.r_forearm_yaw:
        reachy.r_arm.r_forearm_yaw.present_position,

    reachy.r_arm.r_wrist_pitch:
        reachy.r_arm.r_wrist_pitch.present_position,

    reachy.r_arm.r_wrist_roll:
        reachy.r_arm.r_wrist_roll.present_position,
}

for joint, value in B2_SAFE_ALIGNED.items():
    print(joint.name, round(value, 2))

In [ ]:
B2_SAFE_ALIGNED = {
    reachy.r_arm.r_shoulder_pitch:  16.48,
    reachy.r_arm.r_shoulder_roll:    9.45,
    reachy.r_arm.r_arm_yaw:         18.28,
    reachy.r_arm.r_elbow_pitch:   -108.19,
    reachy.r_arm.r_forearm_yaw:     32.61,
    reachy.r_arm.r_wrist_pitch:       5.36,
    reachy.r_arm.r_wrist_roll:       54.32,
}

In [ ]:
B2_SAFE_ALIGNED_POSE = reachy.r_arm.forward_kinematics()

print("B2 SAFE ALIGNED pose:")
print(B2_SAFE_ALIGNED_POSE)

print("\nXYZ:")
print(
    B2_SAFE_ALIGNED_POSE[0,3],
    B2_SAFE_ALIGNED_POSE[1,3],
    B2_SAFE_ALIGNED_POSE[2,3]
)

In [ ]:
B2_LOWER_3CM = B2_SAFE_ALIGNED_POSE.copy()

# Lower 3 cm
B2_LOWER_3CM[2, 3] -= 0.03

print("New target XYZ:")
print(
    B2_LOWER_3CM[0,3],
    B2_LOWER_3CM[1,3],
    B2_LOWER_3CM[2,3]
)

# Calculate IK only -- DO NOT MOVE YET
b2_lower_joints = reachy.r_arm.inverse_kinematics(
    B2_LOWER_3CM
)

print("\nIK solution:")
print(b2_lower_joints)

In [ ]:
B2_LOWER_3CM_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  b2_lower_joints[0],
    reachy.r_arm.r_shoulder_roll:   b2_lower_joints[1],
    reachy.r_arm.r_arm_yaw:         b2_lower_joints[2],
    reachy.r_arm.r_elbow_pitch:     b2_lower_joints[3],
    reachy.r_arm.r_forearm_yaw:     b2_lower_joints[4],
    reachy.r_arm.r_wrist_pitch:     b2_lower_joints[5],
    reachy.r_arm.r_wrist_roll:      b2_lower_joints[6],
}

print("Lowering 3 cm...")

goto(
    goal_positions=B2_LOWER_3CM_JOINTS,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("3 cm descent complete.")

In [ ]:
B2_LOWER_6CM = B2_SAFE_ALIGNED_POSE.copy()

# 6 cm below the original safe/aligned position
B2_LOWER_6CM[2, 3] -= 0.06

print("New target XYZ:")
print(
    B2_LOWER_6CM[0,3],
    B2_LOWER_6CM[1,3],
    B2_LOWER_6CM[2,3]
)

b2_lower_6cm_joints = reachy.r_arm.inverse_kinematics(
    B2_LOWER_6CM
)

print("\nIK solution:")
print(b2_lower_6cm_joints)

In [ ]:
B2_LOWER_6CM_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  b2_lower_6cm_joints[0],
    reachy.r_arm.r_shoulder_roll:   b2_lower_6cm_joints[1],
    reachy.r_arm.r_arm_yaw:         b2_lower_6cm_joints[2],
    reachy.r_arm.r_elbow_pitch:     b2_lower_6cm_joints[3],
    reachy.r_arm.r_forearm_yaw:     b2_lower_6cm_joints[4],
    reachy.r_arm.r_wrist_pitch:     b2_lower_6cm_joints[5],
    reachy.r_arm.r_wrist_roll:      b2_lower_6cm_joints[6],
}

goto(
    goal_positions=B2_LOWER_6CM_JOINTS,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
B2_PICK_TEST = B2_SAFE_ALIGNED_POSE.copy()

# 7 cm below original safe/aligned position
B2_PICK_TEST[2, 3] -= 0.07

print("Target XYZ:")
print(
    B2_PICK_TEST[0,3],
    B2_PICK_TEST[1,3],
    B2_PICK_TEST[2,3]
)

b2_pick_joints = reachy.r_arm.inverse_kinematics(
    B2_PICK_TEST
)

print("\nIK solution:")
print(b2_pick_joints)

In [ ]:
B2_PICK_TEST_JOINTS = {
    reachy.r_arm.r_shoulder_pitch: b2_pick_joints[0],
    reachy.r_arm.r_shoulder_roll:  b2_pick_joints[1],
    reachy.r_arm.r_arm_yaw:        b2_pick_joints[2],
    reachy.r_arm.r_elbow_pitch:    b2_pick_joints[3],
    reachy.r_arm.r_forearm_yaw:    b2_pick_joints[4],
    reachy.r_arm.r_wrist_pitch:    b2_pick_joints[5],
    reachy.r_arm.r_wrist_roll:     b2_pick_joints[6],
}

goto(
    goal_positions=B2_PICK_TEST_JOINTS,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -19.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
# Keep gripper holding the cube
GRIP = -19.0

# 1. Lift to the 6 cm-lower position
lift_6cm = B2_LOWER_6CM_JOINTS.copy()
lift_6cm[reachy.r_arm.r_gripper] = GRIP

goto(
    goal_positions=lift_6cm,
    duration=3.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

# 2. Lift to the 3 cm-lower position
lift_3cm = B2_LOWER_3CM_JOINTS.copy()
lift_3cm[reachy.r_arm.r_gripper] = GRIP

goto(
    goal_positions=lift_3cm,
    duration=3.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

# 3. Return to the safe/aligned position
lift_safe = B2_SAFE_ALIGNED.copy()
lift_safe[reachy.r_arm.r_gripper] = GRIP

goto(
    goal_positions=lift_safe,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
# Open gripper
goto(
    goal_positions={reachy.r_arm.r_gripper: -29.0},
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

# After cube is positioned correctly, close more firmly
goto(
    goal_positions={reachy.r_arm.r_gripper: -10.0},
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
# ============================================
# RESET TO B2 PICKUP POSITION
# Gripper OPEN
# ============================================

B2_PICK = {
    reachy.r_arm.r_shoulder_pitch:  13.2558,
    reachy.r_arm.r_shoulder_roll:    1.8729,
    reachy.r_arm.r_arm_yaw:         21.6144,
    reachy.r_arm.r_elbow_pitch:    -92.7163,
    reachy.r_arm.r_forearm_yaw:     25.1388,
    reachy.r_arm.r_wrist_pitch:      -5.4446,
    reachy.r_arm.r_wrist_roll:       42.6387,
}

# Open gripper
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -29.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

# Move arm back to the known-good B2 pickup position
goto(
    goal_positions=B2_PICK,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("At B2 pickup position with gripper open.")

In [ ]:
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -10.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Gripper closed to -10 degrees.")

In [ ]:
# 1 cm test lift while maintaining grip

B2_LIFT_1CM = {
    reachy.r_arm.r_shoulder_pitch:  13.7216,
    reachy.r_arm.r_shoulder_roll:    2.2885,
    reachy.r_arm.r_arm_yaw:         21.4492,
    reachy.r_arm.r_elbow_pitch:    -95.1662,
    reachy.r_arm.r_forearm_yaw:     25.5934,
    reachy.r_arm.r_wrist_pitch:      -3.5788,
    reachy.r_arm.r_wrist_roll:       43.7703,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=B2_LIFT_1CM,
    duration=3.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("1 cm test lift complete.")

In [ ]:
B2_LIFT_4CM = {
    reachy.r_arm.r_shoulder_pitch:  14.7238,
    reachy.r_arm.r_shoulder_roll:    3.7861,
    reachy.r_arm.r_arm_yaw:         21.1371,
    reachy.r_arm.r_elbow_pitch:   -102.0907,
    reachy.r_arm.r_forearm_yaw:     27.2323,
    reachy.r_arm.r_wrist_pitch:       1.9974,
    reachy.r_arm.r_wrist_roll:       47.2361,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=B2_LIFT_4CM,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
B2_SAFE_WITH_CUBE = {
    reachy.r_arm.r_shoulder_pitch:  16.48,
    reachy.r_arm.r_shoulder_roll:    9.45,
    reachy.r_arm.r_arm_yaw:         18.28,
    reachy.r_arm.r_elbow_pitch:   -108.19,
    reachy.r_arm.r_forearm_yaw:     32.61,
    reachy.r_arm.r_wrist_pitch:       5.36,
    reachy.r_arm.r_wrist_roll:       54.32,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=B2_SAFE_WITH_CUBE,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Cube lifted to B2 safe position.")

In [ ]:
# Get the ACTUAL current pose while holding the cube
current_pose = reachy.r_arm.forward_kinematics()

B3_SAFE = current_pose.copy()

# Move laterally to the B3 center
B3_SAFE[0, 3] = BOARD_POSES["B3"][0, 3]
B3_SAFE[1, 3] = BOARD_POSES["B3"][1, 3]

# IMPORTANT:
# Keep current Z and current orientation unchanged.

print("B3 safe target:")
print(B3_SAFE)

b3_safe_joints = reachy.r_arm.inverse_kinematics(B3_SAFE)

print("\nB3 safe IK:")
print(b3_safe_joints)

In [ ]:
B3_SAFE_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  16.1833,
    reachy.r_arm.r_shoulder_roll:  -27.6047,
    reachy.r_arm.r_arm_yaw:         16.3124,
    reachy.r_arm.r_elbow_pitch:   -109.0954,
    reachy.r_arm.r_forearm_yaw:     -0.8689,
    reachy.r_arm.r_wrist_pitch:       3.0145,
    reachy.r_arm.r_wrist_roll:       45.1639,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=B3_SAFE_JOINTS,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached safe position above B3.")

In [ ]:
B3_SAFE_POSE = reachy.r_arm.forward_kinematics()

B3_LOWER_3CM = B3_SAFE_POSE.copy()
B3_LOWER_3CM[2, 3] -= 0.03

print("Current Z:", B3_SAFE_POSE[2,3])
print("Target Z: ", B3_LOWER_3CM[2,3])

b3_lower_3cm_joints = reachy.r_arm.inverse_kinematics(
    B3_LOWER_3CM
)

print("\nB3 3cm lower IK:")
print(b3_lower_3cm_joints)

In [ ]:
B3_LOWER_3CM_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  15.6202,
    reachy.r_arm.r_shoulder_roll:  -23.6760,
    reachy.r_arm.r_arm_yaw:         18.5317,
    reachy.r_arm.r_elbow_pitch:    -97.4016,
    reachy.r_arm.r_forearm_yaw:      1.7347,
    reachy.r_arm.r_wrist_pitch:      -0.6789,
    reachy.r_arm.r_wrist_roll:       45.5142,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=B3_LOWER_3CM_JOINTS,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("B3 descent complete.")

In [ ]:
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -29.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Cube released at B3.")

In [ ]:
B3_RETREAT = {
    reachy.r_arm.r_shoulder_pitch:  16.1833,
    reachy.r_arm.r_shoulder_roll:  -27.6047,
    reachy.r_arm.r_arm_yaw:         16.3124,
    reachy.r_arm.r_elbow_pitch:   -109.0954,
    reachy.r_arm.r_forearm_yaw:     -0.8689,
    reachy.r_arm.r_wrist_pitch:       3.0145,
    reachy.r_arm.r_wrist_roll:       45.1639,
    reachy.r_arm.r_gripper:         -29.0,
}

goto(
    goal_positions=B3_RETREAT,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Retreated safely above B3.")

In [ ]:
HOME_POSITION = {
    reachy.r_arm.r_shoulder_pitch:   11.56,
    reachy.r_arm.r_shoulder_roll:   -20.17,
    reachy.r_arm.r_arm_yaw:         -12.74,
    reachy.r_arm.r_elbow_pitch:    -115.84,
    reachy.r_arm.r_forearm_yaw:      -3.52,
    reachy.r_arm.r_wrist_pitch:       29.62,
    reachy.r_arm.r_wrist_roll:        32.61,
    reachy.r_arm.r_gripper:          -29.0,
}

goto(
    goal_positions=HOME_POSITION,
    duration=7.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Arm returned to HOME.")

In [ ]:
# Capture SUPPLY SAFE position

print("=== SUPPLY SAFE JOINTS ===")
for joint in [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]:
    print(joint.name, round(joint.present_position, 2))

print("\n=== SUPPLY SAFE CARTESIAN POSE ===")
SUPPLY_SAFE_POSE = reachy.r_arm.forward_kinematics()
print(SUPPLY_SAFE_POSE)

print("\nXYZ:")
print(
    SUPPLY_SAFE_POSE[0,3],
    SUPPLY_SAFE_POSE[1,3],
    SUPPLY_SAFE_POSE[2,3]
)

In [ ]:
SUPPLY_SAFE = {
    reachy.r_arm.r_shoulder_pitch:  13.23,
    reachy.r_arm.r_shoulder_roll:    5.85,
    reachy.r_arm.r_arm_yaw:        -12.04,
    reachy.r_arm.r_elbow_pitch:    -78.93,
    reachy.r_arm.r_forearm_yaw:      2.29,
    reachy.r_arm.r_wrist_pitch:     -27.86,
    reachy.r_arm.r_wrist_roll:      -11.43,
}

In [ ]:
SUPPLY_HIGH_POSE = SUPPLY_SAFE_POSE.copy()

# Raise 5 cm
SUPPLY_HIGH_POSE[2, 3] += 0.05

print("Current Z:", SUPPLY_SAFE_POSE[2,3])
print("Higher Z: ", SUPPLY_HIGH_POSE[2,3])

supply_high_joints = reachy.r_arm.inverse_kinematics(
    SUPPLY_HIGH_POSE
)

print("\nSUPPLY HIGH IK:")
print(supply_high_joints)

In [ ]:
SUPPLY_HIGH = {
    reachy.r_arm.r_shoulder_pitch:  17.7093,
    reachy.r_arm.r_shoulder_roll:   -2.3171,
    reachy.r_arm.r_arm_yaw:         -2.7593,
    reachy.r_arm.r_elbow_pitch:    -93.4187,
    reachy.r_arm.r_forearm_yaw:     -3.4636,
    reachy.r_arm.r_wrist_pitch:     -16.9034,
    reachy.r_arm.r_wrist_roll:      -20.7591,
    reachy.r_arm.r_gripper:         -29.0,
}

goto(
    goal_positions=SUPPLY_HIGH,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached SUPPLY HIGH.")

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

for j in joints:
    print(
        j.name,
        "present =", round(j.present_position, 2),
        "goal =", round(j.goal_position, 2),
        "compliant =", j.compliant
    )

In [ ]:
print("Arm compliant states:")
for j in joints:
    print(j.name, j.compliant)

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

# Synchronize goals with ACTUAL current positions
for j in joints:
    current = j.present_position
    j.goal_position = current
    print(j.name, "synced to", round(current, 2))

print("Goals synchronized.")

In [ ]:
reachy.turn_on("r_arm")

In [ ]:
for j in joints:
    print(
        j.name,
        "present =", round(j.present_position, 2),
        "goal =", round(j.goal_position, 2),
        "compliant =", j.compliant
    )

In [ ]:
SUPPLY_HIGH = {
    reachy.r_arm.r_shoulder_pitch:  17.7093,
    reachy.r_arm.r_shoulder_roll:   -2.3171,
    reachy.r_arm.r_arm_yaw:         -2.7593,
    reachy.r_arm.r_elbow_pitch:    -93.4187,
    reachy.r_arm.r_forearm_yaw:     -3.4636,
    reachy.r_arm.r_wrist_pitch:     -16.9034,
    reachy.r_arm.r_wrist_roll:      -20.7591,
    reachy.r_arm.r_gripper:         -29.0,
}

goto(
    goal_positions=SUPPLY_HIGH,
    duration=7.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("SUPPLY_HIGH move completed.")

In [ ]:
print("=== NEW SUPPLY SAFE JOINTS ===")

joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

for j in joints:
    print(j.name, round(j.present_position, 2))

SUPPLY_SAFE_POSE = reachy.r_arm.forward_kinematics()

print("\n=== NEW SUPPLY SAFE POSE ===")
print(SUPPLY_SAFE_POSE)

print("\nXYZ:")
print(
    SUPPLY_SAFE_POSE[0,3],
    SUPPLY_SAFE_POSE[1,3],
    SUPPLY_SAFE_POSE[2,3]
)

In [ ]:
SUPPLY_SAFE = {
    reachy.r_arm.r_shoulder_pitch:  25.45,
    reachy.r_arm.r_shoulder_roll:   10.24,
    reachy.r_arm.r_arm_yaw:        -25.49,
    reachy.r_arm.r_elbow_pitch:   -108.63,
    reachy.r_arm.r_forearm_yaw:    -11.34,
    reachy.r_arm.r_wrist_pitch:     -10.28,
    reachy.r_arm.r_wrist_roll:       31.46,
}

SUPPLY_SAFE_POSE = np.array([
    [-0.09938067, -0.04742305, -0.99391878,  0.21407864],
    [-0.12210640,  0.99189556, -0.03511725, -0.27951706],
    [ 0.98752899,  0.11787387, -0.10436590, -0.27774352],
    [ 0.0,         0.0,         0.0,         1.0]
])

In [ ]:
SUPPLY_LOWER_3CM = SUPPLY_SAFE_POSE.copy()
SUPPLY_LOWER_3CM[2,3] -= 0.03

supply_lower_3cm_joints = reachy.r_arm.inverse_kinematics(
    SUPPLY_LOWER_3CM
)

print("Target XYZ:")
print(
    SUPPLY_LOWER_3CM[0,3],
    SUPPLY_LOWER_3CM[1,3],
    SUPPLY_LOWER_3CM[2,3]
)

print("\nIK solution:")
print(supply_lower_3cm_joints)

In [ ]:
goto(
    goal_positions={
        **SUPPLY_SAFE,
        reachy.r_arm.r_gripper: -29.0
    },
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("At SUPPLY_SAFE.")

In [ ]:
SUPPLY_LOWER_3CM_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  30.5329,
    reachy.r_arm.r_shoulder_roll:  -26.2207,
    reachy.r_arm.r_arm_yaw:          5.5991,
    reachy.r_arm.r_elbow_pitch:    -99.3388,
    reachy.r_arm.r_forearm_yaw:    -28.0345,
    reachy.r_arm.r_wrist_pitch:    -30.9130,
    reachy.r_arm.r_wrist_roll:      -4.4220,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=SUPPLY_LOWER_3CM_JOINTS,
    duration=7.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("3 cm descent complete.")

In [ ]:
SUPPLY_LOWER_4CM = SUPPLY_SAFE_POSE.copy()

# Total of 4 cm below SUPPLY_SAFE
SUPPLY_LOWER_4CM[2,3] -= 0.04

supply_lower_4cm_joints = reachy.r_arm.inverse_kinematics(
    SUPPLY_LOWER_4CM
)

print("Target XYZ:")
print(
    SUPPLY_LOWER_4CM[0,3],
    SUPPLY_LOWER_4CM[1,3],
    SUPPLY_LOWER_4CM[2,3]
)

print("\nIK solution:")
print(supply_lower_4cm_joints)

In [ ]:
SUPPLY_LOWER_4CM_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  29.6605,
    reachy.r_arm.r_shoulder_roll:  -25.3291,
    reachy.r_arm.r_arm_yaw:          6.0422,
    reachy.r_arm.r_elbow_pitch:    -96.7262,
    reachy.r_arm.r_forearm_yaw:    -27.7672,
    reachy.r_arm.r_wrist_pitch:    -32.4220,
    reachy.r_arm.r_wrist_roll:      -3.4015,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=SUPPLY_LOWER_4CM_JOINTS,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Supply pickup: 4 cm descent complete.")

In [ ]:
SUPPLY_LOWER_7CM = SUPPLY_SAFE_POSE.copy()

# Total 7 cm below SUPPLY_SAFE
SUPPLY_LOWER_7CM[2,3] -= 0.07

supply_lower_7cm_joints = reachy.r_arm.inverse_kinematics(
    SUPPLY_LOWER_7CM
)

print("Target XYZ:")
print(
    SUPPLY_LOWER_7CM[0,3],
    SUPPLY_LOWER_7CM[1,3],
    SUPPLY_LOWER_7CM[2,3]
)

print("\nIK solution:")
print(supply_lower_7cm_joints)

In [ ]:
SUPPLY_LOWER_7CM_JOINTS = {
    reachy.r_arm.r_shoulder_pitch:  26.7973,
    reachy.r_arm.r_shoulder_roll:  -22.5847,
    reachy.r_arm.r_arm_yaw:          7.0522,
    reachy.r_arm.r_elbow_pitch:    -88.4653,
    reachy.r_arm.r_forearm_yaw:    -27.1879,
    reachy.r_arm.r_wrist_pitch:    -37.0485,
    reachy.r_arm.r_wrist_roll:      -0.1528,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=SUPPLY_LOWER_7CM_JOINTS,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Supply descent now 7 cm from safe position.")

In [ ]:
SUPPLY_PICK = {
    reachy.r_arm.r_shoulder_pitch:  26.7973,
    reachy.r_arm.r_shoulder_roll:  -22.5847,
    reachy.r_arm.r_arm_yaw:          7.0522,
    reachy.r_arm.r_elbow_pitch:    -88.4653,
    reachy.r_arm.r_forearm_yaw:    -27.1879,
    reachy.r_arm.r_wrist_pitch:    -37.0485,
    reachy.r_arm.r_wrist_roll:      -0.1528,
}

In [ ]:
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -10.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Supply cube gripped.")

In [ ]:
# First lift to the 4 cm-below-safe position
SUPPLY_LIFT_4CM = {
    reachy.r_arm.r_shoulder_pitch:  29.6605,
    reachy.r_arm.r_shoulder_roll:  -25.3291,
    reachy.r_arm.r_arm_yaw:          6.0422,
    reachy.r_arm.r_elbow_pitch:    -96.7262,
    reachy.r_arm.r_forearm_yaw:    -27.7672,
    reachy.r_arm.r_wrist_pitch:    -32.4220,
    reachy.r_arm.r_wrist_roll:      -3.4015,
    reachy.r_arm.r_gripper:        -10.0,
}

goto(
    goal_positions=SUPPLY_LIFT_4CM,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

# Then lift all the way to SUPPLY_SAFE
SUPPLY_SAFE_WITH_CUBE = SUPPLY_SAFE.copy()
SUPPLY_SAFE_WITH_CUBE[reachy.r_arm.r_gripper] = -10.0

goto(
    goal_positions=SUPPLY_SAFE_WITH_CUBE,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Cube lifted to SUPPLY_SAFE.")

In [ ]:
# Capture ACTUAL current pose while holding cube
current_pose = reachy.r_arm.forward_kinematics()

print("Current supply-safe XYZ:")
print(
    current_pose[0,3],
    current_pose[1,3],
    current_pose[2,3]
)

# Preserve current orientation and height
B2_FROM_SUPPLY_SAFE = current_pose.copy()

# Change only X/Y to center over B2
B2_FROM_SUPPLY_SAFE[0,3] = BOARD_POSES["B2"][0,3]
B2_FROM_SUPPLY_SAFE[1,3] = BOARD_POSES["B2"][1,3]

print("\nB2 target XYZ:")
print(
    B2_FROM_SUPPLY_SAFE[0,3],
    B2_FROM_SUPPLY_SAFE[1,3],
    B2_FROM_SUPPLY_SAFE[2,3]
)

b2_from_supply_joints = reachy.r_arm.inverse_kinematics(
    B2_FROM_SUPPLY_SAFE
)

print("\nB2 transport IK:")
print(b2_from_supply_joints)

In [ ]:
B2_TRANSPORT = {
    reachy.r_arm.r_shoulder_pitch:  25.6048,
    reachy.r_arm.r_shoulder_roll:   32.8522,
    reachy.r_arm.r_arm_yaw:         11.1852,
    reachy.r_arm.r_elbow_pitch:    -86.5555,
    reachy.r_arm.r_forearm_yaw:     27.6130,
    reachy.r_arm.r_wrist_pitch:    -24.9753,
    reachy.r_arm.r_wrist_roll:      -6.2609,
    reachy.r_arm.r_gripper:        -10.0,
}

goto(
    goal_positions=B2_TRANSPORT,
    duration=8.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached position above B2.")

In [ ]:
CURRENT_WITH_CUBE = reachy.r_arm.forward_kinematics()

print("Current pose:")
print(CURRENT_WITH_CUBE)

print("\nCurrent XYZ:")
print(
    CURRENT_WITH_CUBE[0,3],
    CURRENT_WITH_CUBE[1,3],
    CURRENT_WITH_CUBE[2,3]
)

print("\nCurrent joints:")
for j in [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]:
    print(j.name, round(j.present_position, 2))

In [ ]:
CURRENT_WITH_CUBE = reachy.r_arm.forward_kinematics()

TRANSPORT_HIGH_POSE = CURRENT_WITH_CUBE.copy()

# Raise 10 cm
TRANSPORT_HIGH_POSE[2,3] += 0.10

print("Current XYZ:")
print(
    CURRENT_WITH_CUBE[0,3],
    CURRENT_WITH_CUBE[1,3],
    CURRENT_WITH_CUBE[2,3]
)

print("\nTransport-high target XYZ:")
print(
    TRANSPORT_HIGH_POSE[0,3],
    TRANSPORT_HIGH_POSE[1,3],
    TRANSPORT_HIGH_POSE[2,3]
)

transport_high_joints = reachy.r_arm.inverse_kinematics(
    TRANSPORT_HIGH_POSE
)

print("\nTRANSPORT HIGH IK:")
print(transport_high_joints)

In [ ]:
TRANSPORT_HIGH = {
    reachy.r_arm.r_shoulder_pitch:  24.8673,
    reachy.r_arm.r_shoulder_roll:  -19.9925,
    reachy.r_arm.r_arm_yaw:          1.7391,
    reachy.r_arm.r_elbow_pitch:   -107.7985,
    reachy.r_arm.r_forearm_yaw:    -15.4704,
    reachy.r_arm.r_wrist_pitch:      -6.9674,
    reachy.r_arm.r_wrist_roll:       -2.3696,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=TRANSPORT_HIGH,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached TRANSPORT_HIGH.")

In [ ]:
HIGH_POSE = reachy.r_arm.forward_kinematics()

b2_x = BOARD_POSES["B2"][0,3]
b2_y = BOARD_POSES["B2"][1,3]

mid_x = (HIGH_POSE[0,3] + b2_x) / 2
mid_y = (HIGH_POSE[1,3] + b2_y) / 2

MIDPOINT_POSE = HIGH_POSE.copy()
MIDPOINT_POSE[0,3] = mid_x
MIDPOINT_POSE[1,3] = mid_y

print("Current high XYZ:")
print(HIGH_POSE[0,3], HIGH_POSE[1,3], HIGH_POSE[2,3])

print("\nMidpoint XYZ:")
print(
    MIDPOINT_POSE[0,3],
    MIDPOINT_POSE[1,3],
    MIDPOINT_POSE[2,3]
)

midpoint_joints = reachy.r_arm.inverse_kinematics(
    MIDPOINT_POSE
)

print("\nMIDPOINT IK:")
print(midpoint_joints)

In [ ]:
MIDPOINT_WITH_CUBE = {
    reachy.r_arm.r_shoulder_pitch:  25.6941,
    reachy.r_arm.r_shoulder_roll:   10.3973,
    reachy.r_arm.r_arm_yaw:          7.7689,
    reachy.r_arm.r_elbow_pitch:   -103.4159,
    reachy.r_arm.r_forearm_yaw:     12.6476,
    reachy.r_arm.r_wrist_pitch:      -4.1294,
    reachy.r_arm.r_wrist_roll:        0.5341,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=MIDPOINT_WITH_CUBE,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached high midpoint.")

In [ ]:
MID_ACTUAL = reachy.r_arm.forward_kinematics()

B2_HIGH_POSE = MID_ACTUAL.copy()

# B2 center
B2_HIGH_POSE[0,3] = BOARD_POSES["B2"][0,3]
B2_HIGH_POSE[1,3] = BOARD_POSES["B2"][1,3]

# Keep current Z and orientation unchanged

print("Current midpoint XYZ:")
print(
    MID_ACTUAL[0,3],
    MID_ACTUAL[1,3],
    MID_ACTUAL[2,3]
)

print("\nB2 HIGH target XYZ:")
print(
    B2_HIGH_POSE[0,3],
    B2_HIGH_POSE[1,3],
    B2_HIGH_POSE[2,3]
)

b2_high_joints = reachy.r_arm.inverse_kinematics(
    B2_HIGH_POSE
)

print("\nB2 HIGH IK:")
print(b2_high_joints)

In [ ]:
B2_HIGH_WITH_CUBE = {
    reachy.r_arm.r_shoulder_pitch:  25.6276,
    reachy.r_arm.r_shoulder_roll:   31.6426,
    reachy.r_arm.r_arm_yaw:         11.7342,
    reachy.r_arm.r_elbow_pitch:    -88.3229,
    reachy.r_arm.r_forearm_yaw:     35.4077,
    reachy.r_arm.r_wrist_pitch:     -11.4885,
    reachy.r_arm.r_wrist_roll:       -6.1475,
    reachy.r_arm.r_gripper:         -10.0,
}

goto(
    goal_positions=B2_HIGH_WITH_CUBE,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

print("Reached high position above B2.")

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

print("=== TRANSPORT JOINTS ===")

for j in joints:
    print(j.name, round(j.present_position, 2))

print("\n=== TRANSPORT POSE ===")

TRANSPORT_POSE = reachy.r_arm.forward_kinematics()
print(TRANSPORT_POSE)

print("\nXYZ:")
print(
    TRANSPORT_POSE[0,3],
    TRANSPORT_POSE[1,3],
    TRANSPORT_POSE[2,3]
)

In [ ]:
TRANSPORT = {
    reachy.r_arm.r_shoulder_pitch:    3.56,
    reachy.r_arm.r_shoulder_roll:    -2.59,
    reachy.r_arm.r_arm_yaw:          25.22,
    reachy.r_arm.r_elbow_pitch:    -119.44,
    reachy.r_arm.r_forearm_yaw:       0.62,
    reachy.r_arm.r_wrist_pitch:       25.58,
    reachy.r_arm.r_wrist_roll:        35.07,
}

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

for j in joints:
    j.goal_position = j.present_position

reachy.turn_on("r_arm")

print("Arm stiffened at TRANSPORT position.")

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

print("=== B2 SAFE JOINTS ===")
for j in joints:
    print(j.name, round(j.present_position, 2))

print("\n=== B2 SAFE POSE ===")
B2_SAFE_POSE = reachy.r_arm.forward_kinematics()
print(B2_SAFE_POSE)

print("\nXYZ:")
print(
    B2_SAFE_POSE[0,3],
    B2_SAFE_POSE[1,3],
    B2_SAFE_POSE[2,3]
)

In [ ]:
B2_SAFE = {
    reachy.r_arm.r_shoulder_pitch:   -1.54,
    reachy.r_arm.r_shoulder_roll:    -7.07,
    reachy.r_arm.r_arm_yaw:          35.95,
    reachy.r_arm.r_elbow_pitch:    -105.29,
    reachy.r_arm.r_forearm_yaw:      -6.24,
    reachy.r_arm.r_wrist_pitch:       3.78,
    reachy.r_arm.r_wrist_roll:       49.48,
}

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

for j in joints:
    j.goal_position = j.present_position

reachy.turn_on("r_arm")

print("Arm stiffened at B2_SAFE.")

In [ ]:
TRANSPORT_WITH_CUBE = dict(TRANSPORT)
TRANSPORT_WITH_CUBE[reachy.r_arm.r_gripper] = -10.0

goto(
    goal_positions=TRANSPORT_WITH_CUBE,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
B2_SAFE_WITH_CUBE = dict(B2_SAFE)
B2_SAFE_WITH_CUBE[reachy.r_arm.r_gripper] = -10.0

goto(
    goal_positions=B2_SAFE_WITH_CUBE,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] -= 0.02   # 2 cm straight down

down_2cm_joints = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:")
print(current_pose[0,3], current_pose[1,3], current_pose[2,3])

print("\nTarget XYZ:")
print(target_pose[0,3], target_pose[1,3], target_pose[2,3])

print("\n2 cm DOWN IK:")
print(down_2cm_joints)

In [ ]:
DOWN_2CM = {
    reachy.r_arm.r_shoulder_pitch:   7.4590,
    reachy.r_arm.r_shoulder_roll:    2.2761,
    reachy.r_arm.r_arm_yaw:         26.4214,
    reachy.r_arm.r_elbow_pitch:   -100.2610,
    reachy.r_arm.r_forearm_yaw:      4.8197,
    reachy.r_arm.r_wrist_pitch:      1.6885,
    reachy.r_arm.r_wrist_roll:      60.8152,
    reachy.r_arm.r_gripper:        -10.0,
}

goto(
    goal_positions=DOWN_2CM,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] -= 0.02

down_next_joints = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:")
print(current_pose[0,3], current_pose[1,3], current_pose[2,3])

print("\nTarget XYZ:")
print(target_pose[0,3], target_pose[1,3], target_pose[2,3])

print("\nNEXT 2 cm DOWN IK:")
print(down_next_joints)

In [ ]:
DOWN_NEXT_2CM = {
    reachy.r_arm.r_shoulder_pitch:   8.7845,
    reachy.r_arm.r_shoulder_roll:    2.2545,
    reachy.r_arm.r_arm_yaw:         24.7719,
    reachy.r_arm.r_elbow_pitch:    -94.9155,
    reachy.r_arm.r_forearm_yaw:      6.5862,
    reachy.r_arm.r_wrist_pitch:     -0.3918,
    reachy.r_arm.r_wrist_roll:      55.4757,
    reachy.r_arm.r_gripper:        -10.0,
}

goto(
    goal_positions=DOWN_NEXT_2CM,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
for step in range(3):

    current_pose = reachy.r_arm.forward_kinematics()

    target_pose = current_pose.copy()
    target_pose[2, 3] -= 0.01   # 1 cm down

    q = reachy.r_arm.inverse_kinematics(target_pose)

    goal = {
        reachy.r_arm.r_shoulder_pitch: q[0],
        reachy.r_arm.r_shoulder_roll:  q[1],
        reachy.r_arm.r_arm_yaw:        q[2],
        reachy.r_arm.r_elbow_pitch:    q[3],
        reachy.r_arm.r_forearm_yaw:    q[4],
        reachy.r_arm.r_wrist_pitch:    q[5],
        reachy.r_arm.r_wrist_roll:     q[6],
        reachy.r_arm.r_gripper:       -10.0,
    }

    print(f"Step {step+1}: target Z =", target_pose[2,3])
    print("IK =", q)

    goto(
        goal_positions=goal,
        duration=3.0,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

print("=== B2 PLACE JOINTS ===")
for j in joints:
    print(j.name, round(j.present_position, 2))

B2_PLACE_POSE = reachy.r_arm.forward_kinematics()

print("\n=== B2 PLACE POSE ===")
print(B2_PLACE_POSE)

print("\nXYZ:")
print(
    B2_PLACE_POSE[0,3],
    B2_PLACE_POSE[1,3],
    B2_PLACE_POSE[2,3]
)

In [ ]:
B2_PLACE = {
    reachy.r_arm.r_shoulder_pitch:   9.80,
    reachy.r_arm.r_shoulder_roll:    0.66,
    reachy.r_arm.r_arm_yaw:         24.43,
    reachy.r_arm.r_elbow_pitch:    -83.85,
    reachy.r_arm.r_forearm_yaw:      7.91,
    reachy.r_arm.r_wrist_pitch:     -0.09,
    reachy.r_arm.r_wrist_roll:      53.96,
}

In [ ]:
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -29.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
for step in range(3):

    current_pose = reachy.r_arm.forward_kinematics()

    target_pose = current_pose.copy()
    target_pose[2, 3] += 0.01   # 1 cm UP

    q = reachy.r_arm.inverse_kinematics(target_pose)

    goal = {
        reachy.r_arm.r_shoulder_pitch: q[0],
        reachy.r_arm.r_shoulder_roll:  q[1],
        reachy.r_arm.r_arm_yaw:        q[2],
        reachy.r_arm.r_elbow_pitch:    q[3],
        reachy.r_arm.r_forearm_yaw:    q[4],
        reachy.r_arm.r_wrist_pitch:    q[5],
        reachy.r_arm.r_wrist_roll:     q[6],
        reachy.r_arm.r_gripper:       -29.0,
    }

    goto(
        goal_positions=goal,
        duration=3.0,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

    print(f"Up step {step+1} complete")

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

for j in joints:
    print(
        j.name,
        "present =", round(j.present_position, 2),
        "goal =", round(j.goal_position, 2),
        "compliant =", j.compliant
    )

print("\nCurrent XYZ:")
pose = reachy.r_arm.forward_kinematics()
print(pose[0,3], pose[1,3], pose[2,3])

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] += 0.02

q_up = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:")
print(
    current_pose[0,3],
    current_pose[1,3],
    current_pose[2,3]
)

print("\nTarget XYZ:")
print(
    target_pose[0,3],
    target_pose[1,3],
    target_pose[2,3]
)

print("\n2 cm UP IK:")
print(q_up)

In [ ]:
UP_2CM = {
    reachy.r_arm.r_shoulder_pitch:  12.6616,
    reachy.r_arm.r_shoulder_roll:    1.3801,
    reachy.r_arm.r_arm_yaw:         23.6687,
    reachy.r_arm.r_elbow_pitch:    -92.6565,
    reachy.r_arm.r_forearm_yaw:      9.6217,
    reachy.r_arm.r_wrist_pitch:      9.6732,
    reachy.r_arm.r_wrist_roll:      55.1853,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=UP_2CM,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] += 0.06   # 6 cm UP

q_up_6 = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:")
print(
    current_pose[0,3],
    current_pose[1,3],
    current_pose[2,3]
)

print("\nTarget XYZ:")
print(
    target_pose[0,3],
    target_pose[1,3],
    target_pose[2,3]
)

print("\n6 cm UP IK:")
print(q_up_6)

In [ ]:
UP_6CM = {
    reachy.r_arm.r_shoulder_pitch:  14.9106,
    reachy.r_arm.r_shoulder_roll:    3.0012,
    reachy.r_arm.r_arm_yaw:         23.2765,
    reachy.r_arm.r_elbow_pitch:   -104.5593,
    reachy.r_arm.r_forearm_yaw:     13.2187,
    reachy.r_arm.r_wrist_pitch:     22.9666,
    reachy.r_arm.r_wrist_roll:      57.8439,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=UP_6CM,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] += 0.04   # 4 cm UP

q = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:", current_pose[0,3], current_pose[1,3], current_pose[2,3])
print("Target XYZ:", target_pose[0,3], target_pose[1,3], target_pose[2,3])
print("IK:", q)

goal = {
    reachy.r_arm.r_shoulder_pitch: q[0],
    reachy.r_arm.r_shoulder_roll:  q[1],
    reachy.r_arm.r_arm_yaw:        q[2],
    reachy.r_arm.r_elbow_pitch:    q[3],
    reachy.r_arm.r_forearm_yaw:    q[4],
    reachy.r_arm.r_wrist_pitch:    q[5],
    reachy.r_arm.r_wrist_roll:     q[6],
    reachy.r_arm.r_gripper:       -29.0,
}

goto(
    goal_positions=goal,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
B2_EXIT = {
    reachy.r_arm.r_shoulder_pitch:  15.5934,
    reachy.r_arm.r_shoulder_roll:    4.1865,
    reachy.r_arm.r_arm_yaw:         22.9000,
    reachy.r_arm.r_elbow_pitch:   -109.5160,
    reachy.r_arm.r_forearm_yaw:     17.5768,
    reachy.r_arm.r_wrist_pitch:     31.9286,
    reachy.r_arm.r_wrist_roll:      57.9873,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=B2_EXIT,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

for j in joints:
    print(
        j.name,
        "present =", round(j.present_position, 2),
        "goal =", round(j.goal_position, 2),
        "error =", round(j.goal_position - j.present_position, 2),
        "compliant =", j.compliant
    )

pose = reachy.r_arm.forward_kinematics()

print("\nActual XYZ:")
print(pose[0,3], pose[1,3], pose[2,3])

In [ ]:
TRANSPORT_OPEN = {
    reachy.r_arm.r_shoulder_pitch:    3.56,
    reachy.r_arm.r_shoulder_roll:    -2.59,
    reachy.r_arm.r_arm_yaw:          25.22,
    reachy.r_arm.r_elbow_pitch:    -119.44,
    reachy.r_arm.r_forearm_yaw:       0.62,
    reachy.r_arm.r_wrist_pitch:       25.58,
    reachy.r_arm.r_wrist_roll:        35.07,
    reachy.r_arm.r_gripper:          -29.0,
}

goto(
    goal_positions=TRANSPORT_OPEN,
    duration=7.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
HOME_POSITION = {
    reachy.r_arm.r_shoulder_pitch:   11.56,
    reachy.r_arm.r_shoulder_roll:   -20.17,
    reachy.r_arm.r_arm_yaw:         -12.74,
    reachy.r_arm.r_elbow_pitch:    -115.84,
    reachy.r_arm.r_forearm_yaw:      -3.52,
    reachy.r_arm.r_wrist_pitch:      29.62,
    reachy.r_arm.r_wrist_roll:       32.61,
    reachy.r_arm.r_gripper:         -29.0,
}

goto(
    goal_positions=HOME_POSITION,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
SUPPLY_SAFE = {
    reachy.r_arm.r_shoulder_pitch:   25.45,
    reachy.r_arm.r_shoulder_roll:    10.24,
    reachy.r_arm.r_arm_yaw:         -25.49,
    reachy.r_arm.r_elbow_pitch:    -108.63,
    reachy.r_arm.r_forearm_yaw:     -11.34,
    reachy.r_arm.r_wrist_pitch:      -10.28,
    reachy.r_arm.r_wrist_roll:        31.46,
    reachy.r_arm.r_gripper:          -29.0,
}

goto(
    goal_positions=SUPPLY_SAFE,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
pose = reachy.r_arm.forward_kinematics()

print("CURRENT:")
print("X =", pose[0,3])
print("Y =", pose[1,3])
print("Z =", pose[2,3])

print("\nORIGINAL SUPPLY SAFE:")
print("X =",  0.21407864)
print("Y =", -0.27951706)
print("Z =", -0.27774352)

print("\nERROR:")
print("dX =", pose[0,3] - 0.21407864)
print("dY =", pose[1,3] - (-0.27951706))
print("dZ =", pose[2,3] - (-0.27774352))

In [ ]:
SUPPLY_INTERMEDIATE = {
    reachy.r_arm.r_shoulder_pitch:  29.6605,
    reachy.r_arm.r_shoulder_roll:  -25.3291,
    reachy.r_arm.r_arm_yaw:          6.0422,
    reachy.r_arm.r_elbow_pitch:    -96.7262,
    reachy.r_arm.r_forearm_yaw:    -27.7672,
    reachy.r_arm.r_wrist_pitch:    -32.4220,
    reachy.r_arm.r_wrist_roll:      -3.4015,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=SUPPLY_INTERMEDIATE,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] -= 0.02   # 2 cm lower

q = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:")
print(current_pose[0,3], current_pose[1,3], current_pose[2,3])

print("\nTarget XYZ:")
print(target_pose[0,3], target_pose[1,3], target_pose[2,3])

print("\n2 cm LOWER IK:")
print(q)

In [ ]:
SUPPLY_PICK_LOWER = {
    reachy.r_arm.r_shoulder_pitch:  27.7617,
    reachy.r_arm.r_shoulder_roll:  -21.1006,
    reachy.r_arm.r_arm_yaw:          6.5955,
    reachy.r_arm.r_elbow_pitch:    -90.4887,
    reachy.r_arm.r_forearm_yaw:    -27.4067,
    reachy.r_arm.r_wrist_pitch:    -33.2882,
    reachy.r_arm.r_wrist_roll:      -0.5625,
    reachy.r_arm.r_gripper:        -29.0,
}

goto(
    goal_positions=SUPPLY_PICK_LOWER,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -10.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
goto(
    goal_positions={
        reachy.r_arm.r_gripper: -7.0
    },
    duration=2.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
CUBE_GRIP = -7.0

In [ ]:
SUPPLY_PICK_OLD_WITH_CUBE = {
    reachy.r_arm.r_shoulder_pitch:  26.7973,
    reachy.r_arm.r_shoulder_roll:  -22.5847,
    reachy.r_arm.r_arm_yaw:          7.0522,
    reachy.r_arm.r_elbow_pitch:    -88.4653,
    reachy.r_arm.r_forearm_yaw:    -27.1879,
    reachy.r_arm.r_wrist_pitch:    -37.0485,
    reachy.r_arm.r_wrist_roll:      -0.1528,
    reachy.r_arm.r_gripper:         -7.0,
}

goto(
    goal_positions=SUPPLY_PICK_OLD_WITH_CUBE,
    duration=4.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
joints = [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]

for j in joints:
    print(
        j.name,
        "present =", round(j.present_position, 2),
        "goal =", round(j.goal_position, 2),
        "error =", round(j.goal_position - j.present_position, 2),
        "compliant =", j.compliant
    )

pose = reachy.r_arm.forward_kinematics()

print("\nActual XYZ:")
print(pose[0,3], pose[1,3], pose[2,3])

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] += 0.03   # 3 cm straight UP

q = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:")
print(
    current_pose[0,3],
    current_pose[1,3],
    current_pose[2,3]
)

print("\nTarget XYZ:")
print(
    target_pose[0,3],
    target_pose[1,3],
    target_pose[2,3]
)

print("\n3 cm UP IK:")
print(q)

In [ ]:
SUPPLY_UP_3CM = {
    reachy.r_arm.r_shoulder_pitch:  30.3621,
    reachy.r_arm.r_shoulder_roll:  -22.7873,
    reachy.r_arm.r_arm_yaw:          6.1181,
    reachy.r_arm.r_elbow_pitch:    -96.1392,
    reachy.r_arm.r_forearm_yaw:    -28.1199,
    reachy.r_arm.r_wrist_pitch:    -30.1518,
    reachy.r_arm.r_wrist_roll:      -3.1878,
    reachy.r_arm.r_gripper:         -7.0,
}

goto(
    goal_positions=SUPPLY_UP_3CM,
    duration=5.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
pose = reachy.r_arm.forward_kinematics()

print("Actual XYZ:")
print(
    pose[0,3],
    pose[1,3],
    pose[2,3]
)

print("\nJoint tracking:")
for j in [
    reachy.r_arm.r_shoulder_pitch,
    reachy.r_arm.r_shoulder_roll,
    reachy.r_arm.r_arm_yaw,
    reachy.r_arm.r_elbow_pitch,
    reachy.r_arm.r_forearm_yaw,
    reachy.r_arm.r_wrist_pitch,
    reachy.r_arm.r_wrist_roll,
]:
    print(
        j.name,
        "present =", round(j.present_position,2),
        "goal =", round(j.goal_position,2),
        "error =", round(j.goal_position-j.present_position,2)
    )

In [ ]:
current_pose = reachy.r_arm.forward_kinematics()

target_pose = current_pose.copy()
target_pose[2, 3] += 0.06

q = reachy.r_arm.inverse_kinematics(target_pose)

print("Current XYZ:")
print(current_pose[0,3], current_pose[1,3], current_pose[2,3])

print("\nTarget XYZ:")
print(target_pose[0,3], target_pose[1,3], target_pose[2,3])

print("\n6 cm UP IK:")
print(q)

In [ ]:
SUPPLY_UP_6CM = {
    reachy.r_arm.r_shoulder_pitch:  34.8012,
    reachy.r_arm.r_shoulder_roll:  -24.7006,
    reachy.r_arm.r_arm_yaw:          4.1634,
    reachy.r_arm.r_elbow_pitch:   -108.9874,
    reachy.r_arm.r_forearm_yaw:    -29.7127,
    reachy.r_arm.r_wrist_pitch:    -19.9402,
    reachy.r_arm.r_wrist_roll:      -8.2861,
    reachy.r_arm.r_gripper:         -7.0,
}

goto(
    goal_positions=SUPPLY_UP_6CM,
    duration=6.0,
    interpolation_mode=InterpolationMode.MINIMUM_JERK
)

In [ ]:
# Once the fingers are correctly positioned around the cube:
for grip in [-5.0, -2.0, 0.0, 3.0]:
    print("Testing grip:", grip)

    goto(
        goal_positions={reachy.r_arm.r_gripper: grip},
        duration=1.5,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )

    input("Check the cube. Press Enter for tighter grip, or Ctrl+C if secure.")